## Imports

In [144]:
import json
import re
import requests

import pandas as pd
import numpy as np
from dadata import Dadata

from bs4 import BeautifulSoup

# import postgres
import psycopg2

## Functions

In [145]:
def fill_address(value):
    if 'дрес' not in value.lower():
        return np.nan
    else:
        try:
            address = value.split('дрес')[1].split(':')[1].strip()
            if ('назначение' or 'площадь') in address:
                address = ', '.join(map(lambda x: x.strip(), value.split('дрес')[1].split(':')[1].split(',', 5)[:-1]))
            elif not address:
                address = ', '.join(map(lambda x: x.strip(), value.split('дрес')[1].split('у', 1)[1].split(',', 5)[:-1]))
            elif 'общей площадью' in address:
                address =  ', '.join(map(lambda x: x.strip(), value.split('дрес')[1].split('у', 1)[1].split(',', 5)[:-2]))

            while address[0] == ':':
                address = address[1:].strip() 
            return address if len(address.split(',')) > 1 else np.nan
        except IndexError:
            return np.nan

In [146]:
def fillna_cadastr_num(x, y):
    pattern = r'\b\d+:\d+:\d+:\d+\b'
    if pd.isnull(x):
        matches = re.findall(pattern, y)
        if matches:
            return matches[0].strip()
        else:
            return np.nan
    else:
        return x

In [147]:
def fillna_address(address: str, cadastr_num: str) -> str:
    dadata_api = "33a1e9d8e1e0a564c028d7d82e1a908f361b82a8"
    secret_key = "b9b0616923b59bf5179c7108c96bf4b267a9ade1"
    dadata = Dadata(dadata_api, secret_key)

    if pd.isnull(address):
        if not pd.isnull(cadastr_num):
            try:
                result = dadata.find_by_id('address', cadastr_num)[0]['value']
                return result
            except IndexError:
                return np.nan
        else:
            return np.nan
    else:
        return address

In [201]:
def get_coords_by_address(address: str):
    from json_scrapper import YANDEX_API
    import psycopg2
    

    print(address)
    if not pd.isnull(address):
        conn = psycopg2.connect(
            host="localhost",
            database="postgres",
            user="postgres",
            password="admin"
        )
        select_query = 'SELECT * FROM geocoding_results src WHERE src.address = %s'
        
        with conn.cursor() as cur:
            cur.execute(select_query, (address,))
            result = cur.fetchone()

        if result and address == result[1]:
            conn.close()
            return result[2]
        else:
            url = f'https://geocode-maps.yandex.ru/1.x/?apikey={YANDEX_API}&geocode={address}&format=json'
            response = requests.get(url)

            if response.status_code != 200:
                conn.close()
                return np.nan
            else:
                coords = ', '.join(response.json()['response']['GeoObjectCollection']['featureMember'][0]['GeoObject']['Point']['pos'].split()[::-1])
                insert_query = '''
                    INSERT INTO geocoding_results(
                        address, 
                        coords
                    ) values (
                        %s,
                        %s
                    );
                '''
                with conn.cursor() as cur:
                    cur.execute(insert_query, (address, coords))
                    conn.commit()
                    conn.close()
                return coords
    else:
        return np.nan

In [197]:
get_coords_by_address("г. Астрахань, Советский район, ул. Фунтовское шоссе, 8 пом. 1'")

'46.315269, 48.032712'

## Main

In [199]:
data_parts = []

with open(r'json_file_full.json', encoding='utf-8') as f:
    data = pd.read_json(f)

for i in data['content']:
    data_parts.extend(i)

df = pd.DataFrame.from_records(data_parts)

df_columns = [
    'id',
    'lotStatus',
    'biddType',
    'biddForm',
    'lotName',
    'lotDescription',
    'priceMin',
    'biddEndTime',
    'lotImages',
    'category',
    'createDate',
    'characteristics'
]
df = df[df_columns].reset_index(drop=True)

df['biddEndTime'] = pd.to_datetime(df['biddEndTime']).dt.tz_localize(None)
df['createDate'] = pd.to_datetime(df['createDate']).dt.tz_localize(None)
df['biddType'] = df['biddType'].apply(lambda x: x['name'])
df['biddForm'] = df['biddForm'].apply(lambda x: x['name'])
df['category'] = df['category'].apply(lambda x: x['name'])
df['area'] = df['characteristics'].apply(
    lambda x: [i['characteristicValue'] for i in x if i['code'] == 'totalAreaRealty'][0] \
                if len([i['characteristicValue'] for i in x if i['code'] == 'totalAreaRealty']) != 0 \
                else np.nan
)
df['cadastral_number'] = df['characteristics'].apply(
    lambda x: [i['characteristicValue'] for i in x if i['code'] == 'cadastralNumberRealty' \
                if 'characteristicValue' in i.keys()][0] if len([i['characteristicValue'] for i in x if i['code'] == 'cadastralNumberRealty' if 'characteristicValue' in i.keys()]) != 0 \
                else np.nan
)
df['address'] = df['lotDescription'].map(fill_address)
df['cadastral_number'] = df.apply(lambda x: fillna_cadastr_num(x['cadastral_number'], x['lotDescription']), axis=1)
df['address'] = df.apply(lambda x: fillna_address(x['address'], x['cadastral_number']), axis=1)
df['lotImages'] = df['lotImages'].apply(lambda x: ['https://torgi.gov.ru/new/file-store/v1/'+i+'?disposition=inline' for i in x])
df['link'] = df['id'].apply(lambda x: 'https://torgi.gov.ru/new/public/lots/lot/' + x)

df.drop(columns=['characteristics'], inplace=True)

df = df.groupby('address').apply(lambda x: x.loc[x['createDate'].idxmax()]).reset_index(drop=True)

## Координаты по адресу

In [202]:
df['coords'] = df['address'].map(get_coords_by_address)

. Москва, вн.тер.г. муниципальный округ Мещанский, пер Астраханский, д. 5/9
410039, г. Саратов, Ново-Астраханское шоссе, д. 79. Описание объекта
414042, Астраханская обл., г. Астрахань, Трусовский район, ул. Галины Николаевой, д. 12, корп. 2, кв. 9
Астраханская обл, Ахтубинский р-н, село Болхуны, пер Д.Бедного, д 15
Астраханская обл, Володарский р-н, поселок Володарский, ул Садовая, д 40
Астраханская обл, Володарский р-н, село Сизый Бугор, ул Нариманова, д 224
Астраханская обл, Енотаевский р-н, село Ленино, тер фермерское хозяйство Северное, зд 1
Астраханская обл, Икрянинский р-н, село Житное, ул Волжская, соор 6А
Астраханская обл, Икрянинский р-н, село Житное, ул Набережная, д 3А
Астраханская обл, Икрянинский р-н, село Ново-Булгары, ул Тукая, д 2
Астраханская обл, Камызякский р-н, село Образцово-Травино, ул Заводская, д 2
Астраханская обл, Красноярский р-н, село Черемуха, ул Джамбула, д 51
Астраханская обл, Лиманский р-н, село Михайловка, ул Советская, д 161
Астраханская обл, Наримано

IndexError: list index out of range

In [105]:
wex = ', '.join(map(lambda x: x.strip(), recent_df.lotDescription[0].split('дрес')[1].split('у', 1)[1].split(',', 5)[:-1]))
wex.split(':')

['',
 ' ',
 'г. Астрахань, ул. М. Максаковой/ ул. Полякова, 39/10, пом. 180, комн. 7']

In [98]:
fill_address(recent_df.lotDescription[0])

if address
if len > 1


''

# TODO
1. Координаты по адресу;
2. Высоту местности;
3. Координаты Ж/Д станций;
4. ...

In [622]:
# Нашёл ip, по которому можно получить адрес по кадастровому номеру бесплатно
# params = {
#     'Cookie': r"device_uuid=ab876b06-da70-4fdc-9b0c-0dacb725c80c; _ym_uid=1695065687733068195; _ym_d=1695065687; _gid=GA1.2.815736755.1695065687; tmr_lvid=aca5eb5b75aaee2475d9fa1c571f7d0a; tmr_lvidTS=1695065687460; _ym_isad=1; PAPVisitorId=22e96c8aa7f5af17aaa16232d1RPMDPH; _ym_visorc=w; popmechanic_sbjs_migrations=popmechanic_1418474375998%3D1%7C%7C%7C1471519752600%3D1%7C%7C%7C1471519752605%3D1; aprt_last_partner=actionpay; aprt_last_apclick=; aprt_last_apsource=17322767; tmr_detect=1%7C1695066370206; laravel_session=eyJpdiI6IjdaNjdHUFRueVJKZ3V3OTNvZE95K0E9PSIsInZhbHVlIjoiS0h4S0R3aS9mZ0dFaDh6OVRMVEhuOFpDbDRqRGdvYzRGazVIcWZWYmpEWXRkZDBYM0pPbnNxZlN5YmwzL0tGL3o3T1ppUlVvQmMzd2xyYWJpT2Q0MTdWamgxd1BodHI1MUJacnp1c3RxNk9KQTFOZFlTQ0ZyY0pvSEV1anowU0EiLCJtYWMiOiI1MGZkNmYwMTYxZjRkOWZhNTM4YjliZTUwNjNmODg0MWU2ZGUwYzMyODQzMjY1YjdlMzVlMzNiYzI1NWVhMDEwIiwidGFnIjoiIn0%3D; _gat_UA-127986179-1=1; _gat=1; _ga_6QZMB4SDLP=GS1.1.1695065687.1.1.1695066460.60.0.0; _ga=GA1.1.2031855303.1695065687",
#     'User-Agent': r"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
#     'Accept': r"application/json, text/plain, */*"
# }
# example = requests.post(r"https://egrnrstr.ru/search/30:09:070302:687", params=params).json()